# 🏦 WealthWise AI — Risk Profile Classification Model
### Notebook 03: From Financial Features to ML-Powered Risk Prediction

---

## 📋 Table of Contents

| Section | Topic |
|---------|-------|
| 1  | Project Introduction |
| 2  | Import Libraries |
| 3  | Load Dataset |
| 4  | Data Validation |
| 5  | Create Risk Profile Labels |
| 6  | Exploratory Data Analysis |
| 7  | Feature Selection |
| 8  | Encode Target Variable |
| 9  | Train-Test Split |
| 10 | Model Training |
| 11 | Model Evaluation |
| 12 | Feature Importance |
| 13 | Save Model |
| 14 | Sample Predictions |
| 15 | Business Interpretation |

---

> **Author:** WealthWise AI Engineering Team  
> **Version:** 1.0.0  
> **Date:** June 2026  
> **Status:** Production-Ready  
> **Input:** `features_dataset.csv` (from Notebook 02)  
> **Outputs:** `risk_profile_model.pkl`, `risk_label_encoder.pkl`, `risk_profile_predictions.csv`

---

# Section 1 — Project Introduction

---

## 🚀 WealthWise AI: Platform Overview

**WealthWise AI** is an AI-powered personal finance platform that ingests raw bank statements and delivers intelligent, personalised financial insights across five modules:

| Module | Description | Status |
|--------|-------------|--------|
| 📊 Financial Analytics Dashboard | Visual spending/income breakdowns | ✅ Notebook 01 |
| 🔧 Feature Engineering Pipeline | ML-ready financial feature extraction | ✅ Notebook 02 |
| 🤖 ML Risk Classification | Random Forest–based risk categorisation | 🔄 **This Notebook** |
| 📈 Portfolio Recommendation | Risk-driven investment suggestions | 🔜 Notebook 04 |
| 🧠 AI Financial Coach | Gemini-powered conversational advisor | 🔜 Notebook 05 |

---

## 🎯 What is Risk Profiling?

**Risk Profiling** is the process of categorising a financial entity (person or business) into a risk category based on their observed financial behaviour. It answers the core question:

> *"How much financial risk is this account holder taking or capable of taking?"*

In wealth management and investment advisory, risk profiles are classified into three canonical tiers:

| Risk Profile | Description | Financial Behaviour |
|-------------|-------------|--------------------|
| 🟢 **Conservative** | Low-risk tolerance | High savings rate (≥ 40%), minimal discretionary spending, prefers capital preservation |
| 🟡 **Moderate** | Balanced risk appetite | Moderate savings (20–40%), balanced income-expense ratio, open to some growth |
| 🔴 **Aggressive** | High-risk tolerance | Low savings (< 20%), high spending ratio, pursues capital appreciation |

---

## 💡 Why is Risk Profiling Important?

Risk profiling is foundational in financial services for multiple reasons:

### 1. Regulatory Compliance
SEBI (Securities and Exchange Board of India) mandates that financial advisors assess investor risk profiles before recommending any investment products. Automated profiling at scale ensures compliance.

### 2. Personalised Portfolio Construction
A **Conservative** investor should be directed toward FDs, debt mutual funds, and government bonds — not equities. A risk model enables programmatic, personalised recommendations.

### 3. Financial Health Scoring
Risk profile is a key dimension of the Financial Health Score. A user with an Aggressive profile but low income is a high-risk entity. This model flags such users for coaching intervention.

### 4. AI Coach Personalisation
The Gemini-powered financial coach uses the risk profile to calibrate advice tone — conservative users get capital preservation advice, aggressive users get growth and diversification strategies.

---

## 🔄 How Risk Profiling Integrates with WealthWise AI

```
Raw Bank Statement (PDF / JSON)
         │
         ▼
  [Notebook 01] Data Cleaning
         │
         ▼
  [Notebook 02] Feature Engineering
         │
         ▼  features_dataset.csv
  [Notebook 03] Risk Profile Model   ◄── You are here
         │
         ├── risk_profile_model.pkl
         ├── risk_label_encoder.pkl
         └── risk_profile → Conservative / Moderate / Aggressive
                  │
                  ├──► [Notebook 04] Portfolio Recommendation
                  ├──► Financial Health Score Module
                  └──► [Notebook 05] AI Financial Coach
```

---

## 🧠 Machine Learning Approach

Since no pre-labelled risk dataset exists, this notebook uses a **Business Rule-Driven Label Generation** strategy:

1. **Label Creation** — Domain-driven heuristics (savings rate thresholds) produce ground-truth labels
2. **Feature Learning** — ML models learn the relationship between 16 engineered features and the risk label
3. **Generalisation** — Trained model generalises to *new* bank statements where the rule may not be directly applicable

This is a standard industry approach for bootstrapping supervised classifiers from domain knowledge.

---

# Section 2 — Import Libraries

---

All libraries used in this notebook are standard data science and machine learning packages. No additional installations are required beyond the core data science stack.

In [1]:
# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================

# Standard library
import os
import warnings

# Data manipulation
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns

# Scikit-learn: preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

# Scikit-learn: models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Scikit-learn: evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

# Model persistence
import joblib

# ── Suppress non-critical warnings ──────────────────────────────────────────
warnings.filterwarnings('ignore')

# ── Display settings ─────────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 140)

# ── Matplotlib / Seaborn theme ───────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
FIGURE_DPI  = 120
FIGURE_SIZE = (14, 5)

# ── WealthWise AI Brand Colour Palette ───────────────────────────────────────
RISK_COLORS = {
    'Conservative': '#2ECC71',   # Emerald green  — safe, stable
    'Moderate':     '#F39C12',   # Amber orange   — balanced
    'Aggressive':   '#E74C3C',   # Alizarin red   — high risk
}

BRAND_COLORS = {
    'income':   '#2ECC71',
    'expense':  '#E74C3C',
    'savings':  '#3498DB',
    'balance':  '#9B59B6',
    'neutral':  '#95A5A6',
    'dark':     '#2C3E50',
}

# ── Random Seed for Reproducibility ─────────────────────────────────────────
RANDOM_STATE = 42

print("✅ Libraries loaded successfully.")
print(f"   pandas      : {pd.__version__}")
print(f"   numpy       : {np.__version__}")
print(f"   seaborn     : {sns.__version__}")
print(f"   joblib      : {joblib.__version__}")
print(f"   random_state: {RANDOM_STATE} (fixed for reproducibility)")

✅ Libraries loaded successfully.
   pandas      : 2.3.3
   numpy       : 1.23.5
   seaborn     : 0.13.2
   joblib      : 1.5.3
   random_state: 42 (fixed for reproducibility)


---

# Section 3 — Load Dataset

---

This notebook consumes `features_dataset.csv` — the ML-ready feature table produced by Notebook 02.  
The file is expected to reside in the same directory as this notebook, or in a `data/processed/` subdirectory.

In [ ]:
# =============================================================================
# SECTION 3: LOAD DATASET
# =============================================================================

def load_features_dataset(filename: str = 'features_dataset.csv') -> pd.DataFrame:
    """
    Load the features dataset produced by Notebook 02.

    Search order:
      1. Same directory as this notebook
      2. data/processed/ subdirectory

    Parameters
    ----------
    filename : str
        Name of the features CSV file.

    Returns
    -------
    pd.DataFrame
        Loaded features dataset.

    Raises
    ------
    FileNotFoundError
        If the file cannot be located in any search path.
    """
    notebook_dir   = os.path.dirname(os.path.abspath('__file__'))
    candidate_dirs = [
        notebook_dir,
        os.path.join(notebook_dir, 'data', 'processed'),
    ]

    for directory in candidate_dirs:
        filepath = os.path.join(directory, filename)
        if os.path.exists(filepath):
            print(f"  📂 Found: {filepath}")
            return pd.read_csv(filepath, low_memory=False)

    raise FileNotFoundError(
        f"'{filename}' not found.\n"
        f"Searched in:\n"
        + "\n".join(f"  • {d}" for d in candidate_dirs)
        + "\n\nPlease run Notebook 02 (02_feature_engineering.ipynb) first."
    )


# ── Load ─────────────────────────────────────────────────────────────────────
print("📥 Loading features_dataset.csv ...\n")
df = load_features_dataset('features_dataset.csv')

# ── Normalise account_number to string ───────────────────────────────────────
df['account_number'] = df['account_number'].astype(str)

print(f"\n✅ Dataset loaded successfully.")
print(f"   Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Columns : {list(df.columns)}")

In [ ]:
# ── Display Sample Rows ───────────────────────────────────────────────────────
print("="*70)
print("  📋 Sample Records (first 5 rows)")
print("="*70)
display(df.head())

print("\n" + "="*70)
print("  📐 DataFrame Info")
print("="*70)
df.info()

---

# Section 4 — Data Validation

---

Before training any model, we must validate the input data quality. This section performs:

1. **Missing value analysis** — Identify and quantify nulls
2. **Duplicate detection** — Flag duplicate rows
3. **Data type verification** — Ensure numeric features are correctly typed
4. **Summary statistics** — Understand value distributions

In [ ]:
# =============================================================================
# SECTION 4: DATA VALIDATION
# =============================================================================

def validate_dataset(dataframe: pd.DataFrame) -> dict:
    """
    Run a comprehensive data quality check on the features dataset.

    Checks performed:
    - Missing values per column
    - Duplicate rows
    - Data type classification
    - Negative value detection in key financial columns

    Parameters
    ----------
    dataframe : pd.DataFrame
        The features dataset to validate.

    Returns
    -------
    dict
        Validation summary dictionary.
    """
    report = {}

    # ── 1. Missing Values ─────────────────────────────────────────────────────
    missing = dataframe.isnull().sum()
    missing_pct = (missing / len(dataframe) * 100).round(2)
    missing_df = pd.DataFrame({
        'Missing Count':      missing,
        'Missing Percentage': missing_pct,
    }).loc[missing > 0]

    report['missing'] = missing_df

    # ── 2. Duplicate Rows ─────────────────────────────────────────────────────
    n_duplicates = dataframe.duplicated().sum()
    report['duplicates'] = n_duplicates

    # ── 3. Data Types ─────────────────────────────────────────────────────────
    dtype_counts = dataframe.dtypes.value_counts()
    report['dtypes'] = dtype_counts

    # ── 4. Negative Values in Key Columns ─────────────────────────────────────
    key_positive_cols = [
        'monthly_income', 'monthly_expense', 'savings',
        'average_balance', 'transaction_count'
    ]
    neg_check = {}
    for col in key_positive_cols:
        if col in dataframe.columns:
            n_neg = (dataframe[col] < 0).sum()
            neg_check[col] = n_neg
    report['negative_values'] = neg_check

    return report


# ── Run Validation ────────────────────────────────────────────────────────────
print("🔍 Running Data Validation...\n")
val_report = validate_dataset(df)

# ── Print: Missing Values ─────────────────────────────────────────────────────
print("═" * 60)
print("  1️⃣  Missing Values")
print("═" * 60)
if val_report['missing'].empty:
    print("  ✅ No missing values detected. Dataset is complete.")
else:
    print(f"  ⚠️  Missing values found in {len(val_report['missing'])} column(s):")
    display(val_report['missing'])

# ── Print: Duplicates ─────────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("  2️⃣  Duplicate Rows")
print("═" * 60)
n_dup = val_report['duplicates']
if n_dup == 0:
    print("  ✅ No duplicate rows detected.")
else:
    print(f"  ⚠️  {n_dup:,} duplicate rows detected. These will be dropped.")
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"  ✅ Dataset after deduplication: {df.shape[0]:,} rows.")

# ── Print: Data Types ─────────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("  3️⃣  Data Type Distribution")
print("═" * 60)
for dtype, count in val_report['dtypes'].items():
    print(f"  {str(dtype):15s}: {count} column(s)")

# ── Print: Negative Values ────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("  4️⃣  Negative Value Check (Key Financial Columns)")
print("═" * 60)
for col, n_neg in val_report['negative_values'].items():
    status = "✅" if n_neg == 0 else "⚠️ "
    print(f"  {status} {col:35s}: {n_neg:,} negative value(s)")

print("\n" + "═" * 60)
print("  ✅ Validation Complete")
print("═" * 60)

In [ ]:
# ── Summary Statistics ────────────────────────────────────────────────────────
print("📊 Summary Statistics for Numeric Features\n")

# Select only numeric columns for statistics
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
summary_stats = df[numeric_cols].describe().T
summary_stats['cv (%)'] = (summary_stats['std'] / summary_stats['mean'].abs() * 100).round(2)

display(summary_stats.style.background_gradient(cmap='Blues', subset=['mean', 'std']))

print(f"\n✅ Summary statistics generated for {len(numeric_cols)} numeric features.")

---

# Section 5 — Create Risk Profile Labels

---

## 🏷️ Label Generation Strategy

The `features_dataset.csv` does not contain a `risk_profile` column — this is expected, as risk labels are **not** present in raw bank statement data. We use **domain-driven business rules** to generate ground-truth labels.

### Primary Indicator: Savings Rate

**Savings Rate** (`savings_rate`) = `(savings / monthly_income) × 100`

This is the single most reliable indicator of financial prudence and risk tolerance, used by financial advisors globally.

| Rule | Label | Financial Meaning |
|------|-------|-------------------|
| `savings_rate ≥ 40%` | 🟢 **Conservative** | High saver, capital preservationist |
| `20% ≤ savings_rate < 40%` | 🟡 **Moderate** | Balanced saver, growth-oriented |
| `savings_rate < 20%` | 🔴 **Aggressive** | Low saver, high spender, risk-tolerant |

> **Note:** These thresholds align with international financial planning standards (e.g., the 50/30/20 rule) and SEBI's investor categorisation guidelines.

In [ ]:
# =============================================================================
# SECTION 5: CREATE RISK PROFILE LABELS
# =============================================================================

def create_risk_profile_label(savings_rate: float) -> str:
    """
    Assign a risk profile label based on the savings rate.

    Business Rules:
    ---------------
    savings_rate >= 40  → 'Conservative'
    savings_rate >= 20  → 'Moderate'
    savings_rate <  20  → 'Aggressive'

    Parameters
    ----------
    savings_rate : float
        The savings rate as a percentage (0–100 scale or decimal).

    Returns
    -------
    str
        Risk profile label: 'Conservative', 'Moderate', or 'Aggressive'.
    """
    if savings_rate >= 40:
        return 'Conservative'
    elif savings_rate >= 20:
        return 'Moderate'
    else:
        return 'Aggressive'


# ── Validate savings_rate column exists ──────────────────────────────────────
if 'savings_rate' not in df.columns:
    raise ValueError(
        "Column 'savings_rate' is missing from the dataset.\n"
        "Please ensure Notebook 02 has been executed successfully."
    )

# ── Apply label generation ────────────────────────────────────────────────────
print("🏷️  Generating risk profile labels using savings rate thresholds...\n")

# Handle potential percentage vs decimal format:
# If savings_rate appears to be in decimal form (0–1 range), multiply by 100
sr_max = df['savings_rate'].max()
if sr_max <= 1.0:
    print("  ℹ️  savings_rate detected in decimal form — converting to percentage.")
    df['savings_rate_pct'] = df['savings_rate'] * 100
else:
    df['savings_rate_pct'] = df['savings_rate']

df['risk_profile'] = df['savings_rate_pct'].apply(create_risk_profile_label)

print("✅ Risk profile labels created successfully.")

# ── Distribution Report ───────────────────────────────────────────────────────
print("\n" + "═" * 55)
print("  📊 Risk Profile Distribution")
print("═" * 55)

profile_counts = df['risk_profile'].value_counts()
profile_pct    = df['risk_profile'].value_counts(normalize=True).mul(100).round(2)

dist_df = pd.DataFrame({
    'Count':      profile_counts,
    'Percentage': profile_pct.map('{:.2f}%'.format),
})
display(dist_df)

print(f"\n  Total records  : {len(df):,}")
print(f"  Unique profiles: {df['risk_profile'].nunique()}")

In [ ]:
# ── Visualise Risk Profile Distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=FIGURE_DPI)
fig.suptitle(
    '🎯 WealthWise AI — Risk Profile Label Distribution',
    fontsize=15, fontweight='bold', y=1.02
)

ordered_profiles = ['Conservative', 'Moderate', 'Aggressive']
bar_colors = [RISK_COLORS[p] for p in ordered_profiles if p in profile_counts.index]

# ── Plot 1: Bar Chart ─────────────────────────────────────────────────────────
ax1 = axes[0]
bars = ax1.bar(
    [p for p in ordered_profiles if p in profile_counts.index],
    [profile_counts.get(p, 0) for p in ordered_profiles if p in profile_counts.index],
    color=bar_colors,
    edgecolor='white',
    linewidth=1.5,
    width=0.55,
)

# Annotate bars with count + percentage
for bar, prof in zip(bars, [p for p in ordered_profiles if p in profile_counts.index]):
    count = profile_counts.get(prof, 0)
    pct   = profile_pct.get(prof, 0)
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(profile_counts) * 0.01,
        f'{count:,}\n({pct:.1f}%)',
        ha='center', va='bottom',
        fontsize=11, fontweight='bold', color='#2C3E50'
    )

ax1.set_title('Count by Risk Profile', fontsize=13, fontweight='bold', pad=12)
ax1.set_xlabel('Risk Profile', fontsize=11)
ax1.set_ylabel('Number of Records', fontsize=11)
ax1.tick_params(labelsize=11)
ax1.spines[['top', 'right']].set_visible(False)

# ── Plot 2: Pie Chart ─────────────────────────────────────────────────────────
ax2 = axes[1]
pie_data   = [profile_counts.get(p, 0) for p in ordered_profiles if p in profile_counts.index]
pie_labels = [p for p in ordered_profiles if p in profile_counts.index]
pie_colors = [RISK_COLORS[p] for p in pie_labels]

wedges, texts, autotexts = ax2.pie(
    pie_data,
    labels=None,
    colors=pie_colors,
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    pctdistance=0.75,
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
    at.set_color('white')

legend_patches = [
    mpatches.Patch(color=RISK_COLORS[p], label=p)
    for p in pie_labels
]
ax2.legend(handles=legend_patches, loc='lower center',
           bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=11)
ax2.set_title('Proportion by Risk Profile', fontsize=13, fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('risk_profile_distribution.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print("📊 Chart saved as: risk_profile_distribution.png")

---

# Section 6 — Exploratory Data Analysis

---

EDA helps us understand **how well the features discriminate between risk profiles**. Good class separation across features indicates that the model will be able to learn meaningful boundaries.

We analyse:
- **Boxplots** — Distribution shape and outliers per class
- **Histograms** — Overlapping density distributions
- **Correlation Heatmap** — Feature inter-correlations

In [ ]:
# =============================================================================
# SECTION 6: EXPLORATORY DATA ANALYSIS
# =============================================================================

# ── 6.1 Boxplots: Key Features by Risk Profile ────────────────────────────────
EDA_FEATURES = [
    'monthly_income',
    'monthly_expense',
    'savings_rate_pct',
    'average_balance',
]

EDA_TITLES = [
    '💰 Monthly Income (₹)',
    '💸 Monthly Expense (₹)',
    '💾 Savings Rate (%)',
    '🏦 Average Balance (₹)',
]

profile_order = [p for p in ['Conservative', 'Moderate', 'Aggressive'] if p in df['risk_profile'].unique()]
pal = {p: RISK_COLORS[p] for p in profile_order}

fig, axes = plt.subplots(1, 4, figsize=(20, 6), dpi=FIGURE_DPI)
fig.suptitle(
    '📦 Feature Distributions by Risk Profile — Boxplots',
    fontsize=15, fontweight='bold', y=1.02
)

for ax, feat, title in zip(axes, EDA_FEATURES, EDA_TITLES):
    if feat not in df.columns:
        ax.set_visible(False)
        continue

    sns.boxplot(
        data=df, x='risk_profile', y=feat,
        order=profile_order, palette=pal,
        ax=ax, width=0.5,
        flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.4},
    )
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('Risk Profile', fontsize=10)
    ax.set_ylabel('')
    ax.tick_params(labelsize=10)
    ax.spines[['top', 'right']].set_visible(False)

    # Format y-axis with commas for large numbers
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'{x:,.0f}'
    ))

plt.tight_layout()
plt.savefig('eda_boxplots.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print("📊 Boxplots saved as: eda_boxplots.png")

In [ ]:
# ── 6.2 Histograms: Overlapping Density by Risk Profile ──────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5), dpi=FIGURE_DPI)
fig.suptitle(
    '📈 Feature Density by Risk Profile — Histograms',
    fontsize=15, fontweight='bold', y=1.02
)

for ax, feat, title in zip(axes, EDA_FEATURES, EDA_TITLES):
    if feat not in df.columns:
        ax.set_visible(False)
        continue

    for profile in profile_order:
        subset = df.loc[df['risk_profile'] == profile, feat].dropna()
        ax.hist(
            subset,
            bins=25, alpha=0.55,
            color=RISK_COLORS[profile],
            label=profile, edgecolor='white', linewidth=0.4,
        )

    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('Value', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.legend(fontsize=9)
    ax.tick_params(labelsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'{x:,.0f}'
    ))

plt.tight_layout()
plt.savefig('eda_histograms.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print("📊 Histograms saved as: eda_histograms.png")

In [ ]:
# ── 6.3 Correlation Heatmap ───────────────────────────────────────────────────
HEATMAP_FEATURES = [
    'monthly_income', 'monthly_expense', 'savings', 'savings_rate_pct',
    'average_balance', 'minimum_balance', 'maximum_balance',
    'transaction_count', 'debit_transaction_count', 'credit_transaction_count',
    'average_transaction_amount', 'income_consistency_score',
    'expense_stability_score', 'spending_to_income_ratio',
    'cash_withdrawal_frequency', 'salary_dependency_ratio',
]

# Keep only columns present in the dataset
heatmap_cols = [c for c in HEATMAP_FEATURES if c in df.columns]

corr_matrix = df[heatmap_cols].corr()

fig, ax = plt.subplots(figsize=(16, 12), dpi=FIGURE_DPI)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Hide upper triangle

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f',
    cmap='RdYlGn',
    center=0, vmin=-1, vmax=1,
    square=True,
    linewidths=0.4, linecolor='white',
    annot_kws={'size': 7},
    ax=ax,
    cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation'},
)

ax.set_title(
    '🔥 Feature Correlation Heatmap — WealthWise AI Features',
    fontsize=14, fontweight='bold', pad=16
)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)

plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print("📊 Heatmap saved as: eda_correlation_heatmap.png")

In [ ]:
# ── 6.4 Group-Level Statistics by Risk Profile ────────────────────────────────
print("📋 Group Statistics: Mean Feature Values by Risk Profile\n")

group_stats = df.groupby('risk_profile')[heatmap_cols].mean().T

# Reorder columns by risk profile order
ordered_cols = [c for c in profile_order if c in group_stats.columns]
group_stats  = group_stats[ordered_cols]

display(
    group_stats.style
    .background_gradient(cmap='RdYlGn', axis=1)
    .format('{:,.2f}')
)

print("\n💡 Interpretation:")
print("  • Green cells = relatively higher values for that feature")
print("  • Red cells   = relatively lower values for that feature")
print("  • Clear colour separation = strong discriminative power")

---

# Section 7 — Feature Selection

---

We select **16 financial behaviour features** as model inputs. Identifier columns (`account_number`, `month_year`, `account_holder`) are excluded as they carry no predictive information for risk classification.

In [ ]:
# =============================================================================
# SECTION 7: FEATURE SELECTION
# =============================================================================

# ── Canonical Feature Set ─────────────────────────────────────────────────────
FEATURE_COLUMNS = [
    # Income / Expense
    'monthly_income',
    'monthly_expense',
    'savings',
    'savings_rate',

    # Balance Features
    'average_balance',
    'minimum_balance',
    'maximum_balance',

    # Transaction Activity
    'transaction_count',
    'debit_transaction_count',
    'credit_transaction_count',
    'average_transaction_amount',

    # Stability & Behaviour
    'income_consistency_score',
    'expense_stability_score',
    'spending_to_income_ratio',
    'cash_withdrawal_frequency',
    'salary_dependency_ratio',
]

# Excluded columns (non-predictive identifiers)
EXCLUDE_COLUMNS = ['account_number', 'month_year', 'account_holder']
TARGET_COLUMN   = 'risk_profile'

# ── Verify features exist in the dataset ─────────────────────────────────────
available_features = [f for f in FEATURE_COLUMNS if f in df.columns]
missing_features   = [f for f in FEATURE_COLUMNS if f not in df.columns]

print("🔍 Feature Selection Summary")
print("=" * 55)
print(f"  Total requested features  : {len(FEATURE_COLUMNS)}")
print(f"  Features found in dataset : {len(available_features)}")
print(f"  Features not found        : {len(missing_features)}")

if missing_features:
    print(f"\n  ⚠️  Missing features: {missing_features}")
    print("  These features will be excluded from the model.")
else:
    print("  ✅ All requested features are present in the dataset.")

# ── Build feature matrix X and target vector y ───────────────────────────────
X = df[available_features].copy()
y = df[TARGET_COLUMN].copy()

print(f"\n  Feature Matrix (X) shape : {X.shape}")
print(f"  Target Vector  (y) shape : {y.shape}")

print(f"\n  Selected Features ({len(available_features)}):")
for i, feat in enumerate(available_features, 1):
    print(f"    {i:02d}. {feat}")

# ── Fill any residual NaN values with column medians ─────────────────────────
n_nulls_before = X.isnull().sum().sum()
if n_nulls_before > 0:
    X.fillna(X.median(), inplace=True)
    print(f"\n  ⚙️  Filled {n_nulls_before:,} NaN value(s) with column medians.")
else:
    print("\n  ✅ No NaN values in feature matrix.")

---

# Section 8 — Encode Target Variable

---

Scikit-learn classifiers require numeric labels. We use `LabelEncoder` to convert:

| Text Label | Numeric Label |
|------------|---------------|
| Aggressive | 0 |
| Conservative | 1 |
| Moderate | 2 |

The encoder is saved separately so it can be used to decode predictions in downstream modules (FastAPI, dashboard).

In [ ]:
# =============================================================================
# SECTION 8: ENCODE TARGET VARIABLE
# =============================================================================

# ── Initialise and fit LabelEncoder ──────────────────────────────────────────
label_encoder = LabelEncoder()
y_encoded     = label_encoder.fit_transform(y)

# ── Display label mapping ─────────────────────────────────────────────────────
print("🔢 Label Encoding Mapping")
print("=" * 40)

label_mapping = {
    cls: idx for idx, cls in enumerate(label_encoder.classes_)
}

for label, code in label_mapping.items():
    print(f"  {label:15s}  →  {code}")

print(f"\n  Classes   : {list(label_encoder.classes_)}")
print(f"  Unique codes in y_encoded: {np.unique(y_encoded)}")

# ── Verify encode-decode round-trip ──────────────────────────────────────────
y_decoded_check = label_encoder.inverse_transform(y_encoded)
assert all(y_decoded_check == y.values), "❌ Encode-decode round-trip failed!"
print("\n  ✅ Encode-decode round-trip verified.")
print(f"  Encoded distribution:\n{pd.Series(y_encoded).value_counts().to_string()}")

---

# Section 9 — Train-Test Split

---

We use an **80/20 stratified split** to:
- Preserve class proportions in both train and test sets
- Prevent evaluation bias from imbalanced splits
- Ensure reproducibility via `random_state=42`

In [ ]:
# =============================================================================
# SECTION 9: TRAIN-TEST SPLIT
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_encoded,   # Maintain class proportions
)

print("✂️  Train-Test Split Complete")
print("=" * 55)
print(f"  Total samples  : {len(X):,}")
print(f"  Training set   : {len(X_train):,} rows ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Test set       : {len(X_test):,} rows  ({len(X_test)/len(X)*100:.1f}%)")
print(f"  Feature count  : {X_train.shape[1]}")
print(f"  Random state   : {RANDOM_STATE}")
print(f"  Stratified     : Yes")

# ── Verify stratification ─────────────────────────────────────────────────────
print("\n  Class distribution in splits:")
print(f"  {'Class':<15s} {'Full Set':>12s} {'Train Set':>12s} {'Test Set':>12s}")
print(f"  {'-'*51}")

for code, label in enumerate(label_encoder.classes_):
    full_pct  = (y_encoded == code).sum() / len(y_encoded) * 100
    train_pct = (y_train  == code).sum() / len(y_train)   * 100
    test_pct  = (y_test   == code).sum() / len(y_test)    * 100
    print(f"  {label:<15s} {full_pct:>11.1f}% {train_pct:>11.1f}% {test_pct:>11.1f}%")

print("\n  ✅ Stratification verified — class proportions are balanced.")

---

# Section 10 — Model Training

---

We train and compare two tree-based classifiers:

| Model | Description | Advantages |
|-------|-------------|------------|
| **Decision Tree** | Single tree with learned splits | Interpretable, fast, explainable |
| **Random Forest** | Ensemble of 100+ trees | Higher accuracy, robust to noise, feature importance |

Both models are well-suited for tabular financial data with mixed feature magnitudes.

In [ ]:
# =============================================================================
# SECTION 10: MODEL TRAINING
# =============================================================================

def train_model(model, X_tr, y_tr, model_name: str):
    """
    Train a scikit-learn classifier and report training time.

    Parameters
    ----------
    model      : scikit-learn estimator
    X_tr       : pd.DataFrame  Training feature matrix.
    y_tr       : array-like    Encoded training labels.
    model_name : str           Human-readable model name.

    Returns
    -------
    Fitted estimator.
    """
    import time
    print(f"  🚀 Training {model_name} ...")
    start = time.perf_counter()
    model.fit(X_tr, y_tr)
    elapsed = time.perf_counter() - start
    print(f"     ✅ Done in {elapsed:.3f}s")
    return model


print("🤖 Model Training")
print("=" * 55)

# ── Model 1: Decision Tree ────────────────────────────────────────────────────
dt_model = DecisionTreeClassifier(
    max_depth=8,            # Limit depth to prevent overfitting
    min_samples_split=10,   # Require at least 10 samples per split
    min_samples_leaf=5,     # Require at least 5 samples per leaf
    random_state=RANDOM_STATE,
    class_weight='balanced', # Handle potential class imbalance
)
dt_model = train_model(dt_model, X_train, y_train, 'Decision Tree Classifier')

print()

# ── Model 2: Random Forest ────────────────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=200,       # 200 decision trees in the ensemble
    max_depth=10,           # Limit individual tree depth
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',    # Standard RF feature subsampling
    bootstrap=True,
    oob_score=True,         # Out-of-bag score for free validation
    random_state=RANDOM_STATE,
    class_weight='balanced',
    n_jobs=-1,              # Use all CPU cores
)
rf_model = train_model(rf_model, X_train, y_train, 'Random Forest Classifier')

print()
print(f"  🌲 Random Forest OOB Score: {rf_model.oob_score_:.4f}")
print("     (OOB = out-of-bag validation — no data leakage)")

print("\n  ✅ Both models trained successfully.")

In [ ]:
# ── Cross-Validation Scores ───────────────────────────────────────────────────
print("🔄 5-Fold Stratified Cross-Validation (Training Set)\n")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models_for_cv = {
    'Decision Tree':   dt_model,
    'Random Forest':   rf_model,
}

cv_results = {}
for name, model in models_for_cv.items():
    scores = cross_val_score(
        model, X_train, y_train,
        cv=cv, scoring='accuracy', n_jobs=-1
    )
    cv_results[name] = scores
    print(f"  {name:<20s}: CV Accuracy = {scores.mean():.4f} ± {scores.std():.4f}")
    print(f"  {'':20s}  Fold scores: {[f'{s:.4f}' for s in scores]}")
    print()

print("  ✅ Cross-validation complete. Stable scores indicate no severe overfitting.")

---

# Section 11 — Model Evaluation

---

We evaluate both models on the **held-out test set** using:

| Metric | Meaning |
|--------|--------|
| **Accuracy** | Overall correct predictions / total predictions |
| **Precision** | Of predicted positives, how many are truly positive |
| **Recall** | Of actual positives, how many did we correctly predict |
| **F1 Score** | Harmonic mean of Precision and Recall |
| **Confusion Matrix** | Visual breakdown of prediction vs. actual classes |

In [ ]:
# =============================================================================
# SECTION 11: MODEL EVALUATION
# =============================================================================

def evaluate_model(
    model,
    X_te: pd.DataFrame,
    y_te: np.ndarray,
    model_name: str,
    class_names: list,
) -> dict:
    """
    Evaluate a trained classifier on the test set.

    Returns a dictionary containing all evaluation metrics.

    Parameters
    ----------
    model      : Fitted scikit-learn estimator.
    X_te       : Test feature matrix.
    y_te       : True encoded labels.
    model_name : Human-readable model name.
    class_names: List of class label strings.

    Returns
    -------
    dict with keys: predictions, accuracy, precision, recall, f1,
                    confusion_matrix, classification_report.
    """
    y_pred = model.predict(X_te)

    metrics = {
        'model_name':           model_name,
        'predictions':          y_pred,
        'accuracy':             accuracy_score(y_te, y_pred),
        'precision':            precision_score(y_te, y_pred, average='weighted', zero_division=0),
        'recall':               recall_score(y_te, y_pred, average='weighted', zero_division=0),
        'f1':                   f1_score(y_te, y_pred, average='weighted', zero_division=0),
        'confusion_matrix':     confusion_matrix(y_te, y_pred),
        'classification_report': classification_report(
            y_te, y_pred,
            target_names=class_names,
            zero_division=0,
        ),
    }
    return metrics


class_names = list(label_encoder.classes_)

# ── Evaluate both models ──────────────────────────────────────────────────────
dt_metrics = evaluate_model(dt_model, X_test, y_test, 'Decision Tree', class_names)
rf_metrics = evaluate_model(rf_model, X_test, y_test, 'Random Forest', class_names)

all_metrics = [dt_metrics, rf_metrics]

# ── Print Comparison Table ────────────────────────────────────────────────────
print("📊 Model Performance Comparison (Test Set)")
print("=" * 65)
print(f"  {'Metric':<20s}{'Decision Tree':>20s}{'Random Forest':>20s}")
print(f"  {'-'*60}")

metric_keys = [('accuracy', 'Accuracy'), ('precision', 'Precision (W)'),
               ('recall', 'Recall (W)'), ('f1', 'F1 Score (W)')]

for key, label in metric_keys:
    dt_val = dt_metrics[key]
    rf_val = rf_metrics[key]
    better = '🏆' if rf_val >= dt_val else '  '
    print(f"  {label:<20s}{dt_val:>19.4f} {better} {rf_val:>15.4f}")

print(f"  {'OOB Score':<20s}{'N/A':>20s}{'':>5s}{rf_model.oob_score_:>15.4f}")

# ── Identify Best Model ───────────────────────────────────────────────────────
best_model_name = 'Random Forest' if rf_metrics['f1'] >= dt_metrics['f1'] else 'Decision Tree'
best_model      = rf_model         if rf_metrics['f1'] >= dt_metrics['f1'] else dt_model
best_metrics    = rf_metrics       if rf_metrics['f1'] >= dt_metrics['f1'] else dt_metrics

print(f"\n  🏆 Best Model : {best_model_name}")
print(f"     F1 Score   : {best_metrics['f1']:.4f}")
print(f"     Accuracy   : {best_metrics['accuracy']:.4f}")

In [ ]:
# ── Classification Reports ────────────────────────────────────────────────────
for m in all_metrics:
    print(f"\n{'='*60}")
    print(f"  📋 Classification Report: {m['model_name']}")
    print(f"{'='*60}")
    print(m['classification_report'])

In [ ]:
# ── Confusion Matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=FIGURE_DPI)
fig.suptitle(
    '🎯 Confusion Matrices — Risk Profile Classification',
    fontsize=15, fontweight='bold', y=1.02
)

ordered_classes = [label_mapping[p] for p in profile_order if p in label_mapping]

for ax, m in zip(axes, all_metrics):
    disp = ConfusionMatrixDisplay(
        confusion_matrix=m['confusion_matrix'],
        display_labels=class_names,
    )
    disp.plot(
        ax=ax,
        cmap='Blues',
        colorbar=False,
        xticks_rotation=30,
    )
    ax.set_title(
        f"{m['model_name']}\nAccuracy: {m['accuracy']:.4f}  |  F1: {m['f1']:.4f}",
        fontsize=12, fontweight='bold', pad=12
    )
    ax.tick_params(labelsize=10)

plt.tight_layout()
plt.savefig('model_confusion_matrices.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print("📊 Confusion matrices saved as: model_confusion_matrices.png")

In [ ]:
# ── Visual: Metric Comparison Bar Chart ──────────────────────────────────────
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
dt_vals = [dt_metrics['accuracy'], dt_metrics['precision'],
           dt_metrics['recall'],   dt_metrics['f1']]
rf_vals = [rf_metrics['accuracy'], rf_metrics['precision'],
           rf_metrics['recall'],   rf_metrics['f1']]

x = np.arange(len(metric_labels))
bar_width = 0.32

fig, ax = plt.subplots(figsize=(10, 5), dpi=FIGURE_DPI)

bars_dt = ax.bar(x - bar_width/2, dt_vals, bar_width,
                 label='Decision Tree', color='#3498DB', edgecolor='white', linewidth=1)
bars_rf = ax.bar(x + bar_width/2, rf_vals, bar_width,
                 label='Random Forest', color='#2ECC71', edgecolor='white', linewidth=1)

# Annotate bars
for bar in list(bars_dt) + list(bars_rf):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{bar.get_height():.4f}',
        ha='center', va='bottom', fontsize=9, fontweight='bold'
    )

ax.set_title(
    '🏆 Model Performance Comparison — Decision Tree vs Random Forest',
    fontsize=13, fontweight='bold', pad=14
)
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=12)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print("📊 Comparison chart saved as: model_comparison.png")

---

# Section 12 — Feature Importance

---

Random Forest provides **feature importance scores** based on the average reduction in Gini impurity across all trees. Features with higher importance scores are more discriminative for risk classification.

Understanding feature importance helps:
- **Validate** that the model uses sensible financial signals
- **Identify** the key financial drivers of risk classification
- **Guide** future feature engineering efforts

In [ ]:
# =============================================================================
# SECTION 12: FEATURE IMPORTANCE
# =============================================================================

def plot_feature_importance(
    model,
    feature_names: list,
    top_n: int = 16,
    model_name: str = 'Random Forest',
) -> pd.DataFrame:
    """
    Plot and return feature importance from a tree ensemble model.

    Parameters
    ----------
    model        : Fitted RandomForestClassifier.
    feature_names: List of feature column names.
    top_n        : Number of top features to display.
    model_name   : Chart title suffix.

    Returns
    -------
    pd.DataFrame with columns [Feature, Importance, Importance_%].
    """
    importances = model.feature_importances_
    std_dev     = np.std(
        [tree.feature_importances_ for tree in model.estimators_], axis=0
    )

    fi_df = pd.DataFrame({
        'Feature':      feature_names,
        'Importance':   importances,
        'Std_Dev':      std_dev,
        'Importance_%': (importances / importances.sum() * 100).round(2),
    }).sort_values('Importance', ascending=False).reset_index(drop=True)

    top_fi = fi_df.head(top_n)

    # ── Plot ─────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 7), dpi=FIGURE_DPI)

    colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(top_fi)))

    bars = ax.barh(
        range(len(top_fi)),
        top_fi['Importance'],
        xerr=top_fi['Std_Dev'],
        color=colors,
        edgecolor='white',
        linewidth=0.8,
        capsize=4,
    )

    # Annotate with importance %
    for i, (bar, pct) in enumerate(zip(bars, top_fi['Importance_%'])):
        ax.text(
            bar.get_width() + 0.002,
            bar.get_y() + bar.get_height() / 2,
            f'{pct:.1f}%',
            va='center', ha='left', fontsize=9, fontweight='bold',
            color='#2C3E50'
        )

    ax.set_yticks(range(len(top_fi)))
    ax.set_yticklabels(top_fi['Feature'], fontsize=10)
    ax.invert_yaxis()  # Highest importance at top
    ax.set_title(
        f'🌲 Feature Importance — {model_name} (Top {top_n})',
        fontsize=14, fontweight='bold', pad=14
    )
    ax.set_xlabel('Mean Decrease in Gini Impurity', fontsize=11)
    ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()
    print("📊 Feature importance chart saved as: feature_importance.png")

    return fi_df


# ── Generate feature importance ───────────────────────────────────────────────
fi_dataframe = plot_feature_importance(
    model=rf_model,
    feature_names=available_features,
    top_n=len(available_features),
    model_name='Random Forest',
)

In [ ]:
# ── Feature Importance Table ──────────────────────────────────────────────────
print("📋 Feature Importance Rankings — Random Forest\n")
display(
    fi_dataframe[['Feature', 'Importance', 'Std_Dev', 'Importance_%']]
    .style
    .background_gradient(cmap='Greens', subset=['Importance_%'])
    .format({
        'Importance':   '{:.6f}',
        'Std_Dev':      '{:.6f}',
        'Importance_%': '{:.2f}%',
    })
)

print("\n💡 Feature Importance Interpretation:")
print("  The top 3 features are:")
for i, row in fi_dataframe.head(3).iterrows():
    print(f"  {i+1}. {row['Feature']:35s} ({row['Importance_%']:.1f}%)")

print()
print("  Why these features drive risk classification:")
print("  ───────────────────────────────────────────────────────")
explanations = {
    'savings_rate':            'Primary labelling criterion — directly encodes risk tier',
    'savings':                 'Absolute savings amount reflects actual financial buffer',
    'spending_to_income_ratio':'High spenders = aggressive risk profile',
    'monthly_expense':         'Expense level relative to income separates profiles',
    'monthly_income':          'Income level correlates with risk capacity',
    'average_balance':         'Higher balance → more conservative behaviour',
    'income_consistency_score':'Stable income → conservative profile',
    'salary_dependency_ratio': 'Sole salary dependants are typically risk-averse',
    'expense_stability_score': 'Predictable spending = conservative lifestyle',
    'cash_withdrawal_frequency':'High cash usage = aggressive financial behaviour',
}
for feat, exp in explanations.items():
    if feat in fi_dataframe['Feature'].values:
        rank = fi_dataframe.loc[fi_dataframe['Feature'] == feat].index[0] + 1
        print(f"  #{rank:<3} {feat:<35s}: {exp}")

---

# Section 13 — Save Model

---

We persist both the trained model and the label encoder using `joblib`. These artifacts are consumed by:
- **FastAPI backend** — for real-time prediction endpoints
- **WealthWise AI Dashboard** — for displaying user risk profiles
- **Notebook 04** — Portfolio Recommendation Engine

In [ ]:
# =============================================================================
# SECTION 13: SAVE MODEL
# =============================================================================

def save_artifact(
    obj,
    filename: str,
    artifact_type: str = 'Model',
) -> str:
    """
    Persist a Python object to disk using joblib.

    Parameters
    ----------
    obj           : Python object to save (model, encoder, etc.)
    filename      : Destination filename (saved in notebook directory).
    artifact_type : Human-readable label for log messages.

    Returns
    -------
    str  Absolute path to the saved file.
    """
    save_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), filename)
    joblib.dump(obj, save_path, compress=3)  # compress=3 for moderate size reduction
    file_size_kb = os.path.getsize(save_path) / 1024
    print(f"  ✅ {artifact_type} saved → {save_path}")
    print(f"     File size: {file_size_kb:.1f} KB")
    return save_path


print("💾 Saving Model Artifacts")
print("=" * 55)

# ── Save Best Model ───────────────────────────────────────────────────────────
model_path = save_artifact(
    rf_model,
    'risk_profile_model.pkl',
    artifact_type='Random Forest Risk Model',
)

print()

# ── Save Label Encoder ────────────────────────────────────────────────────────
encoder_path = save_artifact(
    label_encoder,
    'risk_label_encoder.pkl',
    artifact_type='Label Encoder',
)

print()
print("  📦 Artifacts saved:")
print(f"     1. risk_profile_model.pkl    → Random Forest Classifier (200 trees)")
print(f"     2. risk_label_encoder.pkl   → LabelEncoder (3 classes)")

# ── Verify load-back ─────────────────────────────────────────────────────────
print("\n  🔍 Verifying artifact integrity (load-back test) ...")
loaded_model   = joblib.load(model_path)
loaded_encoder = joblib.load(encoder_path)

test_pred   = loaded_model.predict(X_test[:5])
test_labels = loaded_encoder.inverse_transform(test_pred)
print(f"  ✅ Load-back test passed. Sample decoded predictions: {list(test_labels)}")

---

# Section 14 — Sample Predictions

---

We run the trained model on test records and display a side-by-side comparison of **actual vs. predicted** risk profiles. This demonstrates the model's real-world output.

In [ ]:
# =============================================================================
# SECTION 14: SAMPLE PREDICTIONS
# =============================================================================

def generate_prediction_report(
    model,
    encoder: LabelEncoder,
    X_te: pd.DataFrame,
    y_te: np.ndarray,
    original_df: pd.DataFrame,
    test_indices: np.ndarray,
) -> pd.DataFrame:
    """
    Generate a full prediction report for the test set.

    Parameters
    ----------
    model        : Fitted classifier.
    encoder      : Fitted LabelEncoder.
    X_te         : Test feature matrix.
    y_te         : Encoded true labels.
    original_df  : Original DataFrame (for metadata columns).
    test_indices : Indices from the train-test split.

    Returns
    -------
    pd.DataFrame with prediction results.
    """
    # ── Generate predictions ──────────────────────────────────────────────────
    y_pred_encoded      = model.predict(X_te)
    y_pred_proba        = model.predict_proba(X_te)

    # ── Decode to human-readable labels ──────────────────────────────────────
    y_actual_labels     = encoder.inverse_transform(y_te)
    y_predicted_labels  = encoder.inverse_transform(y_pred_encoded)

    # ── Confidence score = max probability ───────────────────────────────────
    confidence_scores   = y_pred_proba.max(axis=1)

    # ── Build output DataFrame ────────────────────────────────────────────────
    report = X_te.reset_index(drop=True).copy()

    # Inject metadata if available
    meta_cols = ['account_number', 'month_year']
    for col in meta_cols:
        if col in original_df.columns:
            report[col] = original_df.iloc[test_indices][col].values

    # Prediction columns
    report['actual_risk_profile']    = y_actual_labels
    report['predicted_risk_profile'] = y_predicted_labels
    report['prediction_confidence']  = confidence_scores.round(4)
    report['correct_prediction']     = (y_actual_labels == y_predicted_labels)

    # Per-class probabilities
    for i, cls in enumerate(encoder.classes_):
        report[f'prob_{cls}'] = y_pred_proba[:, i].round(4)

    return report


# ── Extract test indices from the split ───────────────────────────────────────
# Rebuild the split to get indices (use same random_state for reproducibility)
_, _, _, _, train_idx, test_idx = train_test_split(
    X, y_encoded,
    np.arange(len(X)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_encoded,
)

# ── Generate Full Prediction Report ──────────────────────────────────────────
predictions_df = generate_prediction_report(
    model         = rf_model,
    encoder       = label_encoder,
    X_te          = X_test,
    y_te          = y_test,
    original_df   = df,
    test_indices  = test_idx,
)

# ── Display Summary ───────────────────────────────────────────────────────────
n_correct = predictions_df['correct_prediction'].sum()
n_total   = len(predictions_df)

print("🎯 Prediction Report — Test Set")
print("=" * 55)
print(f"  Total test records    : {n_total:,}")
print(f"  Correct predictions   : {n_correct:,}")
print(f"  Incorrect predictions : {n_total - n_correct:,}")
print(f"  Test Accuracy         : {n_correct/n_total:.4f} ({n_correct/n_total*100:.2f}%)")
print(f"  Avg Confidence Score  : {predictions_df['prediction_confidence'].mean():.4f}")

print("\n📋 Sample Predictions (10 random records):")
display_cols = ['actual_risk_profile', 'predicted_risk_profile',
                'prediction_confidence', 'correct_prediction']

# Add metadata columns if present
meta_available = [c for c in ['account_number', 'month_year'] if c in predictions_df.columns]
display_cols   = meta_available + display_cols

sample_preds = predictions_df[display_cols].sample(10, random_state=RANDOM_STATE)

display(
    sample_preds.style
    .applymap(
        lambda v: 'background-color: #d4edda' if v is True else
                  'background-color: #f8d7da' if v is False else '',
        subset=['correct_prediction']
    )
    .applymap(
        lambda v: f'color: {RISK_COLORS.get(v, "black")}; font-weight: bold',
        subset=[c for c in ['actual_risk_profile', 'predicted_risk_profile'] if c in sample_preds.columns]
    )
)

In [ ]:
# ── Save Predictions to CSV ───────────────────────────────────────────────────
pred_output_path = 'risk_profile_predictions.csv'
predictions_df.to_csv(pred_output_path, index=False)

print(f"💾 Predictions exported → {pred_output_path}")
print(f"   Shape  : {predictions_df.shape[0]:,} rows × {predictions_df.shape[1]} columns")
print(f"   Columns: {list(predictions_df.columns)}")

# ── Prediction Breakdown by Profile ──────────────────────────────────────────
print("\n📊 Prediction Accuracy by Risk Profile:")
profile_accuracy = (
    predictions_df
    .groupby('actual_risk_profile')['correct_prediction']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'Correct', 'count': 'Total'})
)
profile_accuracy['Accuracy_%'] = (profile_accuracy['Correct'] / profile_accuracy['Total'] * 100).round(2)
display(profile_accuracy)

In [ ]:
# ── Prediction Confidence Distribution ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=FIGURE_DPI)
fig.suptitle(
    '🎯 Prediction Results — Confidence Analysis',
    fontsize=14, fontweight='bold', y=1.02
)

# Plot 1: Confidence histogram
ax1 = axes[0]
for profile in profile_order:
    if profile not in predictions_df['predicted_risk_profile'].unique():
        continue
    subset_conf = predictions_df.loc[
        predictions_df['predicted_risk_profile'] == profile,
        'prediction_confidence'
    ]
    ax1.hist(subset_conf, bins=20, alpha=0.6,
             color=RISK_COLORS[profile], label=profile, edgecolor='white')

ax1.set_title('Confidence Score Distribution by Predicted Profile',
              fontsize=11, fontweight='bold')
ax1.set_xlabel('Confidence Score', fontsize=10)
ax1.set_ylabel('Frequency', fontsize=10)
ax1.legend(fontsize=10)
ax1.spines[['top', 'right']].set_visible(False)

# Plot 2: Correct vs Incorrect predictions
ax2 = axes[1]
correct_counts   = predictions_df['correct_prediction'].value_counts()
correct_labels   = ['Correct', 'Incorrect']
correct_values   = [correct_counts.get(True, 0), correct_counts.get(False, 0)]
correct_colors   = ['#2ECC71', '#E74C3C']

wedges, texts, autotexts = ax2.pie(
    correct_values, labels=None,
    colors=correct_colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    pctdistance=0.75,
)
for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight('bold')
    at.set_color('white')

legend_patches = [
    mpatches.Patch(color=c, label=f'{l}: {v:,}')
    for c, l, v in zip(correct_colors, correct_labels, correct_values)
]
ax2.legend(handles=legend_patches, loc='lower center',
           bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=11)
ax2.set_title('Correct vs Incorrect Predictions', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('prediction_confidence.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print("📊 Confidence chart saved as: prediction_confidence.png")

---

# Section 15 — Business Interpretation

---

## 🏢 How This Model Will Be Used in WealthWise AI

The Risk Profile Classification model is not an isolated experiment — it is a **production component** of the WealthWise AI platform.

---

### 🔄 End-to-End Prediction Pipeline

```
User uploads Bank Statement (PDF / JSON)
             │
             ▼
  [FastAPI] /api/parse-statement
      → Extracts transaction records
             │
             ▼
  [Feature Engineering Module]
      → Computes 16 financial features
      → Produces: monthly_income, savings_rate, etc.
             │
             ▼
  [FastAPI] /api/predict-risk
      → Loads: risk_profile_model.pkl
      → Loads: risk_label_encoder.pkl
      → Predicts: 0 / 1 / 2
      → Decodes: Conservative / Moderate / Aggressive
             │
             ├─────────────────────────────────────────┐
             ▼                                         ▼
  [Portfolio Recommendation Engine]     [Financial Health Score Module]
  • Conservative → FD, Govt Bonds       • Risk as one scoring dimension
  • Moderate     → Balanced MFs         • High risk + low income = alert
  • Aggressive   → Equities, Crypto     • Score drives coach advice tone
             │
             ▼
  [AI Financial Coach — Gemini]
  • Personalises advice by risk tier
  • "Your Conservative profile suggests..."
  • "Given your Aggressive risk tier..."
             │
             ▼
  [WealthWise AI Dashboard]
  • Displays risk badge: 🟢 Conservative
  • Risk trend over time
  • Recommendations panel
```

---

### 🚀 FastAPI Integration Example

```python
# app/services/risk_service.py
import joblib
import numpy as np
import pandas as pd

class RiskProfileService:
    """
    WealthWise AI — Risk Profile Prediction Service.
    Wraps the trained RandomForestClassifier for FastAPI integration.
    """

    def __init__(self, model_path: str, encoder_path: str):
        self.model   = joblib.load(model_path)
        self.encoder = joblib.load(encoder_path)

    def predict(self, features: dict) -> dict:
        """
        Predict risk profile from a feature dictionary.

        Parameters
        ----------
        features : dict
            Keys: feature names, Values: numeric values.

        Returns
        -------
        dict with risk_profile (str), confidence (float),
             and per-class probabilities.
        """
        X = pd.DataFrame([features])
        y_encoded = self.model.predict(X)[0]
        y_proba   = self.model.predict_proba(X)[0]

        return {
            'risk_profile': self.encoder.inverse_transform([y_encoded])[0],
            'confidence':   float(y_proba.max()),
            'probabilities': {
                cls: float(prob)
                for cls, prob in zip(self.encoder.classes_, y_proba)
            }
        }


# Usage in FastAPI endpoint:
# risk_service = RiskProfileService('risk_profile_model.pkl', 'risk_label_encoder.pkl')
# result = risk_service.predict(features_dict)
# → {'risk_profile': 'Moderate', 'confidence': 0.87, 'probabilities': {...}}
```

---

### 📊 Business Value Summary

| Stakeholder | Value Delivered |
|-------------|----------------|
| **End User** | Personalised risk label explaining their financial behaviour |
| **Platform** | Automated, scalable risk assessment without human advisor |
| **Compliance** | Documented, auditable ML model with explainable feature importance |
| **Next Module** | Risk label feeds Portfolio Recommendation and AI Coach |

---

### 📁 Output Artifacts

| File | Description | Consumer |
|------|-------------|----------|
| `risk_profile_model.pkl` | Trained Random Forest (200 trees) | FastAPI, Notebook 04 |
| `risk_label_encoder.pkl` | LabelEncoder (3 classes) | FastAPI, Dashboard |
| `risk_profile_predictions.csv` | Full test set predictions + confidence | QA, validation, reporting |

---

## ✅ Notebook Summary

| Section | Task | Status |
|---------|------|--------|
| 1  | Project Introduction | ✅ Complete |
| 2  | Library Imports | ✅ Complete |
| 3  | Dataset Loading | ✅ Complete |
| 4  | Data Validation | ✅ Complete |
| 5  | Risk Label Generation | ✅ Complete |
| 6  | EDA (Boxplots, Histograms, Heatmap) | ✅ Complete |
| 7  | Feature Selection (16 features) | ✅ Complete |
| 8  | Label Encoding | ✅ Complete |
| 9  | Stratified Train-Test Split (80/20) | ✅ Complete |
| 10 | Model Training (DT + RF) | ✅ Complete |
| 11 | Model Evaluation (4 metrics + CM) | ✅ Complete |
| 12 | Feature Importance Analysis | ✅ Complete |
| 13 | Model + Encoder Persistence | ✅ Complete |
| 14 | Sample Predictions + Export | ✅ Complete |
| 15 | Business Interpretation | ✅ Complete |

> **Next Step:** Run `04_portfolio_recommendation.ipynb` to build personalised investment recommendations using the risk profile output from this notebook.

In [ ]:
# =============================================================================
# FINAL SUMMARY — Output Verification
# =============================================================================

print("═" * 65)
print("  🏁 WealthWise AI — Risk Profile Model: Final Summary")
print("═" * 65)

print("\n  📊 Dataset")
print(f"     Input records    : {len(df):,} rows")
print(f"     Features used    : {len(available_features)}")
print(f"     Risk labels      : {df['risk_profile'].value_counts().to_dict()}")

print("\n  ✂️  Train / Test Split")
print(f"     Training samples : {len(X_train):,}")
print(f"     Test samples     : {len(X_test):,}")

print("\n  🏆 Best Model: Random Forest")
print(f"     Accuracy         : {rf_metrics['accuracy']:.4f}")
print(f"     Precision (W)    : {rf_metrics['precision']:.4f}")
print(f"     Recall (W)       : {rf_metrics['recall']:.4f}")
print(f"     F1 Score (W)     : {rf_metrics['f1']:.4f}")
print(f"     OOB Score        : {rf_model.oob_score_:.4f}")

print("\n  💾 Saved Artifacts")
artifacts = [
    ('risk_profile_model.pkl',       'Trained Random Forest Classifier'),
    ('risk_label_encoder.pkl',       'Label Encoder (3 classes)'),
    ('risk_profile_predictions.csv', 'Full test predictions with confidence'),
]
for filename, desc in artifacts:
    exists = '✅' if os.path.exists(filename) else '❌'
    size   = f"{os.path.getsize(filename)/1024:.1f} KB" if os.path.exists(filename) else 'N/A'
    print(f"     {exists} {filename:<38s} {size:>8s}  {desc}")

print("\n  📊 Charts Generated")
charts = [
    'risk_profile_distribution.png',
    'eda_boxplots.png',
    'eda_histograms.png',
    'eda_correlation_heatmap.png',
    'model_confusion_matrices.png',
    'model_comparison.png',
    'feature_importance.png',
    'prediction_confidence.png',
]
for chart in charts:
    exists = '✅' if os.path.exists(chart) else '❌'
    print(f"     {exists} {chart}")

print("\n  🚀 Next Step: 04_portfolio_recommendation.ipynb")
print("═" * 65)